In [4]:
import pandas as pd

In [7]:
#Q1
df_order = pd.read_csv(r'/home/ducvu0904/Downloads/Datathon/data/orders.csv')

df_order.columns = df_order.columns.str.strip()

df_order['order_date'] = pd.to_datetime(df_order['order_date'], errors='coerce')
df_order = df_order.dropna(subset=['order_date'])

order_counts = df_order.groupby('customer_id').size()
multi_order_customers = order_counts[order_counts > 1].index

df_order_multi = df_order[df_order['customer_id'].isin(multi_order_customers)].copy()

df_order_multi = df_order_multi.sort_values(by=['customer_id', 'order_date'])
df_order_multi['inter_order_days'] = df_order_multi.groupby('customer_id')['order_date'].diff().dt.days

median_gap = df_order_multi['inter_order_days'].median()
print(f'Median gap between orders for customers with multiple orders: {median_gap} days')

Median gap between orders for customers with multiple orders: 144.0 days


In [15]:
#Q2
df_product = pd.read_csv(r'/home/ducvu0904/Downloads/Datathon/data/products.csv')

df_product['profit_margin'] = (df_product['price'] - df_product['cogs']) / df_product['price']

segment_margin = df_product.groupby('segment')['profit_margin'].mean()

highest_margin_segment = segment_margin.idxmax()
highest_margin_value = segment_margin.max()
print(f'Segment with the highest average profit margin: {highest_margin_segment} ({highest_margin_value:.2%})')

Segment with the highest average profit margin: Standard (31.34%)


In [ ]:
#Q3
df_return = pd.read_csv(r'/home/ducvu0904/Downloads/Datathon/data/returns.csv')

df_merged_return_product = pd.merge(df_return, df_product, on='product_id')

df_streetwear = df_merged_return_product[df_merged_return_product['category'] == 'Streetwear']
reason_counts = df_streetwear['return_reason'].value_counts()

top_reason = reason_counts.idxmax()
top_reason_count = reason_counts.max()
print(f'Most common return reason for Streetwear category: {top_reason} ({top_reason_count} returns)')

Most common return reason for Streetwear category: wrong_size (7626 returns)


In [ ]:
#Q4
df_web_traffic = pd.read_csv(r'/home/ducvu0904/Downloads/Datathon/data/web_traffic.csv')

avg_bounce_rate = df_web_traffic.groupby('traffic_source')['bounce_rate'].mean()
lowest_bounce_rate_source = avg_bounce_rate.idxmin()
lowst_bounce_rate_value = avg_bounce_rate.min()
print(f'Lowest bounce rate source: {lowest_bounce_rate_source} with a bounce rate of {lowst_bounce_rate_value:.5f}%')

Lowest bounce rate source: email_campaign with a bounce rate of 0.00446%


In [ ]:
#Q5
df_order_items = pd.read_csv(r'/home/ducvu0904/Downloads/Datathon/data/order_items.csv', low_memory=False)

null_counts = df_order_items['promo_id'].isnull().sum()
total_counts = len(df_order_items)

promo_percentage = (total_counts - null_counts) / total_counts * 100
print(f'Percentage of order items with promo: {promo_percentage:.2f}%')

Percentage of order items with promo: 38.66%


In [ ]:
#Q6
df_customer = pd.read_csv(r'/home/ducvu0904/Downloads/Datathon/data/customers.csv')
df_customer = df_customer.dropna(subset=['age_group'])

df_merged_customer_order = pd.merge(df_customer, df_order, on='customer_id')
age_group_stats = df_merged_customer_order.groupby('age_group').agg(
    total_orders=('order_id', 'count'),
    total_customers=('customer_id', 'nunique')
)
age_group_stats['avg_orders_per_customer'] = age_group_stats['total_orders'] / age_group_stats['total_customers']

highest_group = age_group_stats['avg_orders_per_customer'].idxmax()
highest_value = age_group_stats['avg_orders_per_customer'].max()
print(f'Age group with the highest average orders per customer: {highest_group} ({highest_value:.2f} orders/customer)')

Age group with the highest average orders per customer: 55+ (7.27 orders/customer)


In [ ]:
#Q7
df_geo = pd.read_csv(r'/home/ducvu0904/Downloads/Datathon/data/geography.csv')
df_payment = pd.read_csv(r'/home/ducvu0904/Downloads/Datathon/data/payments.csv')
df_order.columns = df_order.columns.str.strip()
df_geo.columns = df_geo.columns.str.strip()

df_order_payment = pd.merge(df_order, df_payment[['order_id', 'payment_value']], on='order_id')
df_order_payment_geo = pd.merge(df_order_payment, df_geo[['zip', 'region']], on='zip')

display(df_order_payment_geo.head())
result = df_order_payment_geo.groupby('region')['payment_value'].sum().sort_values(ascending=False)
print(result)

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,payment_value,region
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,7967.54,East
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search,71163.75,East
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,33660.99,East
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,53196.25,East
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign,1597.84,East


region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: payment_value, dtype: float64


In [ ]:
#Q8
df_order['order_status'] = df_order['order_status'].str.strip()
df_order['payment_method'] = df_order['payment_method'].str.strip()

cancelled_df = df_order[df_order['order_status'] == 'cancelled']

result = cancelled_df['payment_method'].value_counts()

print(f"Cancelled orders by payment method:\n{result}")

Cancelled orders by payment method:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


In [ ]:
#Q9
df_products = pd.read_csv(r'/home/ducvu0904/Downloads/Datathon/data/products.csv')

df_items_sized = pd.merge(df_order_items, df_products[['product_id', 'size']], on='product_id', how='left')
total_purchases_by_size = df_items_sized.groupby('size').size()

df_return_sized = pd.merge(df_return, df_products[['product_id', 'size']], on='product_id', how='left')
total_returns_by_size = df_return_sized.groupby('size').size()

return_rate_by_size = (total_returns_by_size / total_purchases_by_size * 100).sort_values(ascending=False)

print("Return rate by size:")
print(return_rate_by_size)
print(f"Size with the highest return rate: {return_rate_by_size.idxmax()} ({return_rate_by_size.max():.2f}%)")

Return rate by size:
size
S     5.651527
L     5.624978
M     5.566010
XL    5.520010
dtype: float64
Size with the highest return rate: S (5.65%)


In [ ]:
#Q10
avg_payment = df_payment.groupby("installments")['payment_value'].mean().sort_values(ascending=False)

top_installments = avg_payment.idxmax()
top_value = avg_payment.max()
print(f"Installments with the highest average payment value: {top_installments} ({top_value:.2f})")

Installments with the highest average payment value: 6 (24446.65)
